# Design-8a controlled MiniLM validation

This notebook validates **Runtime v5a — Design-8 facet-local specificity promotion plus explicit-negation veto**.

The only intended runtime change from Design-8 v1 is polarity handling in the **secondary-promotion** path. Primary Top-1 ranking and the frozen Design-7.5b educator-input subsystem remain unchanged.

Prediction order is fixed: **15-case regression → educator-input 24 prediction → frozen 59 prediction → freeze hashes → upload gold → evaluate**. Gold is absent during prediction.


In [ ]:
!pip -q install sentence-transformers scikit-learn pandas


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess, json, csv, hashlib, os, sys

WORK=Path('/content/design8a_validation')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
print('Upload these 3 files now:')
print('  1) Hybrid_KG_Runtime_v5a_Design8a_NegationAwareFacetPromotion.zip')
print('  2) Neo4j_Lean_CSV_Bundle_v1.1.zip')
print('  3) Design8a_Blind_Validation_Harness_v1.zip')
uploaded=files.upload()
print('Uploaded:', list(uploaded))


In [ ]:
def find_uploaded(prefix):
    matches=[Path('/content')/n for n in uploaded if n.startswith(prefix)]
    if not matches: raise FileNotFoundError(prefix)
    return matches[0]

runtime_zip=find_uploaded('Hybrid_KG_Runtime_v5a_Design8a')
repo_zip=find_uploaded('Neo4j_Lean_CSV_Bundle_v1.1')
blind_zip=find_uploaded('Design8a_Blind_Validation_Harness_v1')

runtime_root=WORK/'runtime_pkg'; repo_dir=WORK/'repo'; blind_root=WORK/'blind_pkg'; out=WORK/'outputs'
for d in [runtime_root,repo_dir,blind_root,out]: d.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(runtime_zip) as z: z.extractall(runtime_root)
with zipfile.ZipFile(repo_zip) as z: z.extractall(repo_dir)
with zipfile.ZipFile(blind_zip) as z: z.extractall(blind_root)
RUNTIME=runtime_root/'runtime'
BLIND=blind_root/'blind'
print('Runtime:',RUNTIME)
print('Repo:',repo_dir)
print('Blind harness:',BLIND)


## Stage 1 — Frozen 15-case MiniLM regression guard


In [ ]:
def run(cmd):
    print(' '.join(map(str,cmd)))
    subprocess.run([str(x) for x in cmd],check=True)

reg_pred=out/'regression15_predictions_design8a_minilm.csv'
run(['python',RUNTIME/'hybrid_runtime_v5.py','--repo-dir',repo_dir,'--inputs',RUNTIME/'15_case_inputs_FROZEN_v1.csv','--output',reg_pred,'--backend','sentence-transformer'])
reg_eval=out/'regression15_evaluation_design8a_minilm.csv'
run(['python',RUNTIME/'evaluate_revalidation_design8.py','--predictions',reg_pred,'--gold',RUNTIME/'15_case_gold_FROZEN_v1.csv','--output',reg_eval])
regm=json.load(open(str(reg_eval)+'.metrics.json'))
print(json.dumps(regm,indent=2))
# Regression gates: preserve the validated legacy behaviour and RV11 multi-label recovery.
assert regm['overall_prototype_pass']==15, 'STOP: 15-case legacy regression failed'
assert regm['required_top3_recall_pass']==15, 'STOP: regression Top-3 failed'
assert regm['forbidden_top1_pass']==15, 'STOP: forbidden Top-1 regression'
assert regm['resolution_pass']==15, 'STOP: resolution regression'
rows={r['Test_ID']:r for r in csv.DictReader(open(reg_eval,encoding='utf-8'))}
assert rows['RV11']['Multi_Label_Promotion_Pass']=='PASS', 'STOP: RV11 multi-label regression'
assert rows['RV10']['Forbidden_Promoted_Pass']=='PASS', 'STOP: RV10 SCR_0033 boundary regression'
print('15-case regression gate: PASS')


## Stage 2 — Blind educator-input 24 prediction
Gold is still absent.


In [ ]:
ei_pred=out/'predictions_educator_input_design8a_minilm.csv'
run(['python',BLIND/'run_educator_input_validation_design8a.py','--runtime-dir',RUNTIME,'--repo-dir',repo_dir,'--adapter-file',BLIND/'educator_input_adapter_v3.py','--inputs',BLIND/'educator_input_validation_inputs_blind_v1.csv','--output',ei_pred,'--backend','sentence-transformer'])


## Stage 3 — Blind frozen 59 prediction
The Design-7.5b input layer is not used to alter the frozen 59 benchmark inputs.


In [ ]:
b59_pred=out/'predictions_full_benchmark_design8a_minilm.csv'
run(['python',BLIND/'run_full_benchmark_design8a.py','--runtime-dir',RUNTIME,'--repo-dir',repo_dir,'--inputs',BLIND/'benchmark_inputs_blind_59.csv','--output',b59_pred,'--backend','sentence-transformer'])


## Stage 4 — Freeze predictions before gold is loaded


In [ ]:
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
freeze={
 'regression15_predictions_sha256':sha256(reg_pred),
 'educator24_predictions_sha256':sha256(ei_pred),
 'benchmark59_predictions_sha256':sha256(b59_pred),
 'gold_present_during_prediction':False
}
json.dump(freeze,open(out/'prediction_freeze_manifest.json','w'),indent=2)
print(json.dumps(freeze,indent=2))


## Stage 5 — Upload frozen gold harness
Only now upload **Design8a_Gold_Validation_Harness_FROZEN_v1.zip**.


In [ ]:
gold_upload=files.upload()
gold_name=next((n for n in gold_upload if n.startswith('Design8a_Gold_Validation_Harness_FROZEN_v1')),None)
if not gold_name: raise FileNotFoundError('Design8a_Gold_Validation_Harness_FROZEN_v1.zip')
gold_root=WORK/'gold_pkg'; gold_root.mkdir(exist_ok=True)
with zipfile.ZipFile(Path('/content')/gold_name) as z: z.extractall(gold_root)
GOLD=gold_root/'gold'
print('Gold harness loaded after prediction freeze:',GOLD)


## Stage 6 — Evaluate educator-input regression and frozen 59 benchmark


In [ ]:
ei_eval_dir=out/'educator_eval'; ei_eval_dir.mkdir(exist_ok=True)
run(['python',GOLD/'evaluate_educator_input_validation_v2.py','--predictions',ei_pred,'--gold',GOLD/'educator_input_validation_gold_FROZEN_v1.csv','--context-quality-gold',GOLD/'context_candidate_quality_gold_FROZEN_v1.csv','--output-dir',ei_eval_dir])

b59_eval_dir=out/'benchmark59_eval'; b59_eval_dir.mkdir(exist_ok=True)
run(['python',GOLD/'evaluate_full_benchmark_design8.py','--predictions',b59_pred,'--gold',GOLD/'benchmark_gold_59_FROZEN.csv','--pairs',GOLD/'counterfactual_pairs_FROZEN.csv','--out-dir',b59_eval_dir])


## Stage 7 — Gate summary and Design-7 comparison


In [ ]:
eim=json.load(open(ei_eval_dir/'metrics_educator_input_v2.json'))
bm=json.load(open(b59_eval_dir/'metrics_full_benchmark.json'))
all59=bm['benchmark_sets']['ALL59']

print('EDUCATOR-INPUT REGRESSION')
for k in ['routing_bundle_pass_rate','end_to_end_pass_rate','recommendation_promoted_required_rate','context_candidate_quality_pass_rate','context_candidate_off_scope_avoidance_rate']:
    print(f'{k}: {eim.get(k)}')

BASE7={
 'ready_row_overall_pass_rate':0.717391304347826,
 'top1_required_pass_rate':0.8260869565217391,
 'top3_required_set_pass_rate':0.8260869565217391,
 'promoted_set_precision':0.7105263157894737,
 'promoted_set_recall':0.5869565217391305,
 'promoted_set_f1':0.6428571428571428,
 'strict_end_to_end_pass_rate':0.6521739130434783,
 'multi_label_promotion_pass_rate':0.26666666666666666,
 'unsupported_inference_rate_ready_rows':0.0,
 'counterfactual_direction_pass_rate':1.0
}
DESIGN8_V1={
 'promoted_set_precision':0.6818181818181818,
 'promoted_set_recall':0.6521739130434783,
 'promoted_set_f1':0.6666666666666666,
 'strict_end_to_end_pass_rate':0.6304347826086957,
 'forbidden_promoted_violations':1
}
print()
print('ALL59: Design-7 -> Design-8a')
for k,v in BASE7.items():
    nv=all59.get(k)
    print(f'{k}: {v:.4f} -> {nv:.4f}  delta={nv-v:+.4f}')
print('forbidden_promoted_violations:',all59.get('forbidden_promoted_violations'))
print('max_promoted_count:',all59.get('max_promoted_count'))

ei_gate=(eim.get('routing_bundle_pass_rate')==1.0 and eim.get('end_to_end_pass_rate')==1.0 and eim.get('recommendation_promoted_required_rate')==1.0 and eim.get('context_candidate_quality_pass_rate')==1.0 and eim.get('context_candidate_off_scope_avoidance_rate')==1.0)
print()
print('Educator-input frozen gate:', 'PASS' if ei_gate else 'FAIL')

def ss(s): return {x.strip() for x in (s or '').split(';') if x.strip()}
negation_overlap=[]
for pred_path in [ei_pred,b59_pred]:
    for r in csv.DictReader(open(pred_path,encoding='utf-8')):
        overlap=ss(r.get('Promoted_SCR_IDs')) & ss(r.get('Negation_Vetoed_SCR_IDs'))
        if overlap: negation_overlap.append((r['Test_ID'],sorted(overlap)))
print('Negation-veto/promoted overlap:',negation_overlap)

assert ei_gate, 'STOP: educator-input frozen regression failed'
assert all59.get('forbidden_promoted_violations')==0, 'STOP: forbidden SCR promoted in frozen59'
assert all59.get('unsupported_inference_rate_ready_rows')==0.0, 'STOP: unsupported inference regression'
assert all59.get('counterfactual_direction_pass_rate')==1.0, 'STOP: counterfactual direction regression'
assert not negation_overlap, 'STOP: a negation-vetoed SCR was still promoted'
print('Design-8a safety gates: PASS')

for path,ids in [(b59_pred,{'TC004','XD014','XD021'}),(ei_pred,{'EI011'})]:
    rows={r['Test_ID']:r for r in csv.DictReader(open(path,encoding='utf-8'))}
    for tid in ids:
        r=rows[tid]
        print(tid,'top1=',r.get('Recommended_SCR_ID'),'promoted=',r.get('Promoted_SCR_IDs'),'negative_facets=',r.get('Negative_Promotion_Facets_JSON'),'vetoed=',r.get('Negation_Vetoed_SCR_IDs'))


## Stage 8 — Package results


In [ ]:
run_manifest={
 'release':'Design-8a-v1-controlled-MiniLM',
 'runtime':'v5a-design8a-negation-aware-facet-promotion',
 'model':'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dimension':384,
 'gold_loaded_after_prediction_freeze':True,
 'prediction_freeze_manifest':freeze,
 'educator_input_gate_pass':ei_gate,
 'negation_overlap_count':len(negation_overlap),
 'design7_baseline_embedded_for_comparison':BASE7,
 'design8_v1_reference':DESIGN8_V1
}
json.dump(run_manifest,open(out/'design8a_run_manifest.json','w'),indent=2)
result_zip=Path('/content/Design8a_52Units_59Executions_MiniLM_v1_Results.zip')
if result_zip.exists(): result_zip.unlink()
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(out))
print(result_zip, result_zip.stat().st_size, 'bytes')
files.download(str(result_zip))
